In [ ]:
````xml
<!-- filepath: c:\Users\00113844\Documents\SCIE3314_APSIM_Tutorial\APSIM_Analysis_Notebook.ipynb -->
<VSCode.Cell language="markdown">
# APSIM Output Analysis Notebook
**SCIE3314: Agricultural Production Systems**

This notebook provides Python code to analyze APSIM simulation outputs. Students can use this to:
- Load APSIM exported CSV/Excel files
- Calculate summary statistics
- Create visualizations
- Calculate agronomic metrics (NUE, Harvest Index, water use efficiency)

**Usage:**
1. Export your APSIM simulation results to CSV or Excel format
2. Upload the file to this Colab session (use the folder icon on the left)
3. Update the filename in the code cells below
4. Run each cell sequentially (Shift+Enter)

**Getting Help:**
- Ask AI assistants (Gemini, ChatGPT, Claude) to help interpret results or modify code
- Refer to the APSIM tutorial PDF for context on variables and metrics
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Import Required Libraries
</VSCode.Cell>

<VSCode.Cell language="python">
# Import pandas for data manipulation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Load APSIM Output Data

**Instructions:**
- Export your APSIM results to CSV from the DataStore (right-click → Export → CSV)
- Upload the file using the folder icon (left sidebar)
- Update the filename in the cell below
</VSCode.Cell>

<VSCode.Cell language="python">
# Replace 'your_file.csv' with your actual filename
filename = 'Exercise_A_Results.csv'

# Read the CSV file
df = pd.read_csv(filename)

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Display basic information
print(f"✓ Loaded {len(df)} rows of data")
print(f"✓ Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\n✓ Available columns:")
print(df.columns.tolist())

# Show first few rows
print(f"\n✓ First 5 rows:")
df.head()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Data Inspection and Quality Checks
</VSCode.Cell>

<VSCode.Cell language="python">
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Summary statistics
print("\n" + "="*60)
print("Summary Statistics:")
print("="*60)
df.describe()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Time Series Visualization

### 4.1 Soil Water Balance (ESW over time)
</VSCode.Cell>

<VSCode.Cell language="python">
# Plot ESW (Extractable Soil Water) over time
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['ESW'], linewidth=1.2, color='steelblue', label='ESW')
plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Sowing threshold (100 mm)')
plt.xlabel('Date', fontsize=12)
plt.ylabel('ESW (mm)', fontsize=12)
plt.title('Fallow Water Balance: Extractable Soil Water Over Time', fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate percentage of time above sowing threshold
days_above_threshold = (df['ESW'] > 100).sum()
total_days = len(df)
pct_sowable = (days_above_threshold / total_days) * 100
print(f"\n✓ Days above sowing threshold: {days_above_threshold} ({pct_sowable:.1f}% of simulation)")
</VSCode.Cell>

<VSCode.Cell language="markdown">
### 4.2 Rainfall Analysis
</VSCode.Cell>

<VSCode.Cell language="python">
# Check if Rainfall column exists
if 'Rainfall' in df.columns or 'Rain' in df.columns:
    rain_col = 'Rainfall' if 'Rainfall' in df.columns else 'Rain'
    
    # Extract year from date
    df['Year'] = df['Date'].dt.year
    
    # Calculate annual rainfall
    annual_rainfall = df.groupby('Year')[rain_col].sum()
    
    # Plot annual rainfall
    plt.figure(figsize=(12, 6))
    annual_rainfall.plot(kind='bar', color='dodgerblue', edgecolor='black')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Annual Rainfall (mm)', fontsize=12)
    plt.title('Annual Rainfall Summary', fontsize=14, fontweight='bold')
    plt.axhline(y=annual_rainfall.mean(), color='red', linestyle='--', 
                label=f'Mean: {annual_rainfall.mean():.0f} mm', linewidth=2)
    plt.legend(fontsize=11)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Mean annual rainfall: {annual_rainfall.mean():.1f} mm")
    print(f"✓ Min annual rainfall: {annual_rainfall.min():.1f} mm ({annual_rainfall.idxmin()})")
    print(f"✓ Max annual rainfall: {annual_rainfall.max():.1f} mm ({annual_rainfall.idxmax()})")
    print(f"✓ Coefficient of variation: {(annual_rainfall.std() / annual_rainfall.mean() * 100):.1f}%")
else:
    print("⚠ Rainfall/Rain column not found in dataset")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Monthly Averages
</VSCode.Cell>

<VSCode.Cell language="python">
# Create YearMonth period for grouping
df['YearMonth'] = df['Date'].dt.to_period('M')

# Calculate monthly average ESW
monthly_esw = df.groupby('YearMonth')['ESW'].mean()

print("Monthly Average ESW:")
print(monthly_esw.head(12))

# Plot monthly ESW trend
plt.figure(figsize=(14, 6))
monthly_esw.plot(color='darkorange', linewidth=2)
plt.xlabel('Year-Month', fontsize=12)
plt.ylabel('Average ESW (mm)', fontsize=12)
plt.title('Monthly Average Extractable Soil Water', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6. Agronomic Metrics

### 6.1 Nitrogen Use Efficiency (NUE)

**Formula:** NUE = (Yield_N - Yield_0) / N_rate

**Note:** This calculation requires data from nitrogen response simulations
</VSCode.Cell>

<VSCode.Cell language="python">
# Example NUE calculation (modify with your actual column names and data)
# Assumes you have multiple simulations with different N rates

def calculate_nue(yield_n, yield_0, n_rate):
    """
    Calculate Nitrogen Use Efficiency
    
    Parameters:
    -----------
    yield_n : float
        Yield with N treatment (kg/ha)
    yield_0 : float
        Yield with 0 N (kg/ha)
    n_rate : float
        N application rate (kg/ha)
    
    Returns:
    --------
    float : NUE (kg grain per kg N)
    """
    if n_rate == 0:
        return np.nan
    return (yield_n - yield_0) / n_rate

# Example usage:
# For a simulation with 80 kg N/ha that yielded 3500 kg/ha grain
# and a control (0 N) that yielded 2000 kg/ha:
yield_0 = 2000  # kg/ha (modify with your data)
yield_80 = 3500  # kg/ha (modify with your data)
n_rate = 80  # kg/ha

nue = calculate_nue(yield_80, yield_0, n_rate)
print(f"\n✓ NUE at {n_rate} kg N/ha: {nue:.2f} kg grain/kg N")
print(f"  Interpretation: Every 1 kg of N produced {nue:.1f} kg of additional grain")

# Typical NUE ranges for dryland wheat
print("\n✓ Typical NUE ranges:")
print("  - Excellent: 15-20 kg/kg")
print("  - Good: 12-15 kg/kg")
print("  - Moderate: 8-12 kg/kg")
print("  - Low: <8 kg/kg")
</VSCode.Cell>

<VSCode.Cell language="markdown">
### 6.2 Harvest Index

**Formula:** HI = Grain Yield / Total Above-Ground Biomass

Typical wheat HI: 0.40-0.50 (40-50% of biomass goes to grain)
</VSCode.Cell>

<VSCode.Cell language="python">
# Check if biomass and grain yield columns exist
if 'GrainYield' in df.columns and 'Biomass' in df.columns:
    # Filter to harvest dates (where GrainYield > 0)
    harvest_data = df[df['GrainYield'] > 0].copy()
    
    if len(harvest_data) > 0:
        # Calculate Harvest Index
        harvest_data['HarvestIndex'] = harvest_data['GrainYield'] / harvest_data['Biomass']
        
        print(f"\n✓ Harvest Index Statistics:")
        print(f"  Mean HI: {harvest_data['HarvestIndex'].mean():.3f}")
        print(f"  Min HI: {harvest_data['HarvestIndex'].min():.3f}")
        print(f"  Max HI: {harvest_data['HarvestIndex'].max():.3f}")
        
        # Plot Harvest Index over years
        harvest_data['Year'] = harvest_data['Date'].dt.year
        plt.figure(figsize=(12, 6))
        plt.bar(harvest_data['Year'], harvest_data['HarvestIndex'], 
                color='forestgreen', edgecolor='black', alpha=0.8)
        plt.axhline(y=0.45, color='red', linestyle='--', 
                    label='Typical wheat HI (0.45)', linewidth=2)
        plt.xlabel('Year', fontsize=12)
        plt.ylabel('Harvest Index', fontsize=12)
        plt.title('Harvest Index by Year', fontsize=14, fontweight='bold')
        plt.ylim(0, 0.6)
        plt.legend(fontsize=11)
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠ No harvest events found (GrainYield = 0 for all rows)")
else:
    print("⚠ GrainYield or Biomass columns not found")
    print("Available columns:", df.columns.tolist())
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7. Water Use Efficiency

**Formula:** WUE = Grain Yield / Crop Transpiration

Typical dryland wheat WUE: 15-20 kg grain/mm water
</VSCode.Cell>

<VSCode.Cell language="python">
# Check if transpiration data exists
if 'Transpiration' in df.columns and 'GrainYield' in df.columns:
    # Sum transpiration for each growing season
    df['Year'] = df['Date'].dt.year
    seasonal_data = df.groupby('Year').agg({
        'Transpiration': 'sum',
        'GrainYield': 'max'  # Assumes GrainYield is cumulative and peaks at harvest
    })
    
    # Calculate WUE
    seasonal_data['WUE'] = seasonal_data['GrainYield'] / seasonal_data['Transpiration']
    
    print("\n✓ Water Use Efficiency by Year:")
    print(seasonal_data)
    
    # Plot WUE
    plt.figure(figsize=(12, 6))
    seasonal_data['WUE'].plot(kind='bar', color='teal', edgecolor='black', alpha=0.8)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('WUE (kg grain/mm water)', fontsize=12)
    plt.title('Water Use Efficiency', fontsize=14, fontweight='bold')
    plt.axhline(y=seasonal_data['WUE'].mean(), color='red', linestyle='--',
                label=f"Mean: {seasonal_data['WUE'].mean():.1f} kg/mm", linewidth=2)
    plt.legend(fontsize=11)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Transpiration or GrainYield columns not found")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 8. Export Processed Data

Save your calculated metrics to a new CSV file
</VSCode.Cell>

<VSCode.Cell language="python">
# Export summary statistics to CSV
output_filename = 'APSIM_Analysis_Summary.csv'

# Create summary dataframe (modify based on your analysis)
summary_df = df.groupby('Year').agg({
    'ESW': ['mean', 'min', 'max'],
    'Rainfall': 'sum' if 'Rainfall' in df.columns else lambda x: 0
})

# Flatten column names
summary_df.columns = ['_'.join(col).strip() for col in summary_df.columns.values]

# Save to CSV
summary_df.to_csv(output_filename)
print(f"\n✓ Summary data exported to: {output_filename}")
print("\nPreview:")
print(summary_df.head())
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 9. Additional Resources

**APSIM Documentation:**
- Official docs: https://apsimnextgeneration.netlify.app/
- APSIM Initiative: https://www.apsim.info/

**Python Data Analysis:**
- pandas tutorial: https://pandas.pydata.org/docs/getting_started/index.html
- matplotlib gallery: https://matplotlib.org/stable/gallery/index.html

**AI Assistance:**
- Ask Gemini/ChatGPT/Claude to help interpret results
- Example prompt: "I have APSIM wheat simulation data with ESW values ranging from 50-200 mm. What does this tell me about water availability?"

---

**Tutorial Version:** 1.0 (May 2026)  
**Course:** SCIE3314 - Agricultural Production Systems  
**Institution:** The University of Western Australia
</VSCode.Cell>
````